In [32]:
import pandas as pd
import sqlite3
import os
import datetime
import re
from multidata import casematch, ingest, manifest
from dotenv import load_dotenv
import json

load_dotenv("../.env")

DATA_DIR = "../data/audio"
MANIFEST_PATH = "../manifest.sqlite"
HF_TOKEN = os.getenv('HF_TOKEN')

DEFAULT_PROMPT = (
    "Clinical consultation transcript between a healthcare provider and a patient "
    "discussing medical symptoms, history, diagnosis, and treatment plan, followed "
    "by feedback between the provider and their instructor. Formal medical "
    "terminology is used throughout."
)
filename = '20260226-155705-v309393-20.mp4'
case_id = casematch.resolve(os.path.join(DATA_DIR, filename), manifest_path=MANIFEST_PATH)
case_id

'261498'

In [33]:
audio_filename = f'{case_id}_audio.wav'
import whisperx, gc, torch
device="cpu"

In [34]:
model = whisperx.load_model("large-v3", device, compute_type="int8",
asr_options={"initial_prompt":DEFAULT_PROMPT})


2026-07-28 16:40:14 - whisperx.asr - INFO - No language specified, language will be detected for each audio file (increases inference time)
2026-07-28 16:40:14 - whisperx.vads.pyannote - INFO - Performing voice activity detection using Pyannote...


Lightning automatically upgraded your loaded checkpoint from v1.5.4 to v2.6.5. To apply the upgrade to your files permanently, run `python -m lightning.pytorch.utilities.upgrade_checkpoint ../../../../../opt/homebrew/Caskroom/miniforge/base/envs/md-speech/lib/python3.11/site-packages/whisperx/assets/pytorch_model.bin`


In [35]:
# 20 minute video: 3:38 to transcribe on CPU
audio = whisperx.load_audio(os.path.join(DATA_DIR,audio_filename))
result = model.transcribe(audio, batch_size=8)

2026-07-28 16:41:08 - whisperx.asr - INFO - Detected language: en (0.86) in first 30s of audio


In [36]:
align_model, meta = whisperx.load_align_model(result["language"], device)
result = whisperx.align(result["segments"], align_model, meta, audio, device)

In [37]:
#10:51 (?) on cpu
# 1:34 (!) on mps
dia = whisperx.diarize.DiarizationPipeline(token=HF_TOKEN, device='mps')
diarize_segments = dia(os.path.join(DATA_DIR,audio_filename), max_speakers=4)                 # optionally min/max speakers
result = whisperx.assign_word_speakers(diarize_segments, result)

2026-07-28 16:44:18 - whisperx.diarize - INFO - Loading diarization model: pyannote/speaker-diarization-community-1


In [38]:
out_path = os.path.join("../data/transcripts", f'{case_id}_transcript_diarized.json')
with open(out_path, "w") as f:
    json.dump(result, f, indent=4)
print(f"Wrote {out_path} ")

Wrote ../data/transcripts/261498_transcript_diarized.json 


In [39]:
from multidata.elan import build_eaf

In [40]:
build_eaf(out_path, f'../data/raw/{filename}', f'../data/elan/{case_id}.eaf')

File created successfully: ../data/elan/261498.eaf (684 words added)
Identified Tiers: ['default', 'SPEAKER_02', 'Unknown_Speaker', 'SPEAKER_03', 'SPEAKER_01', 'SPEAKER_00']


'../data/elan/261498.eaf'

In [41]:
from multidata.acoustics import extract_features
extract_features(os.path.join(DATA_DIR,audio_filename),
f'../data/praat/{case_id}.praat')

Data successfully exported to ../data/praat/261498.praat


'../data/praat/261498.praat'